<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/miniprojects/MP8.2B/MP8.2B_heat_capacity_and_coefficient.ipynb)

*Open this notebook in Google Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU). Its first code cell fetches the library files from the public course repository.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# MP8.2B · Heat Capacity and Coefficient Together

*Mini project of the course **Deep Learning for Engineering**, Part 2, in one notebook of five parts:*

**A** The Problem · **B** The Data and the Physics · **C** The Reference Model and the Ground Truth · **D** The Physics-Informed Network · **E** The Report

*Parts A to C are given and solved: read them, run them, and use what they build. Parts D and E are yours: the TODO cells.*

In [ ]:
# files-cell (mini project) ----------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
EXERCISE = (['course_core.py', 'pinn_core.py', 'heatsink.py'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex08-heatsink-cooling/")
PROJECT = (['mp82b.py', 'MP8.2B_truth.npz'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/miniprojects/MP8.2B/")
ON_COLAB = "google.colab" in sys.modules
for files, url in (EXERCISE, PROJECT):
    for f in files:
        if ON_COLAB or not os.path.exists(f):
            if not ON_COLAB and os.path.exists(os.path.join("..", "..", "exercises", "Ex08-heatsink-cooling", f)):
                import shutil; shutil.copy(os.path.join("..", "..", "exercises", "Ex08-heatsink-cooling", f), f)
            else:
                urllib.request.urlretrieve(url + f, f)
            print("fetched", f)
for _m in ['course_core', 'pinn_core', 'heatsink', 'mp82b']:
    sys.modules.pop(_m, None)
print("files ready")

In [ ]:
# --- setup ---------------------------------------------------------------
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import course_core as cc
import heatsink as hs
import mp82b as mp
torch.set_default_dtype(torch.float64)                   # a second derivative of a network needs double precision
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # the GPU, when the runtime has one
cc.OUTPUT_DIR = "MP8.2B_outputs"                          # where the report and its figures are written
print("device:", DEVICE, "| torch", torch.__version__)

def middle(fn, reps=5):
    """The middle of five wall-clock timings of fn(), after one warm-up run;
    on a GPU, synchronised so the time is the computation's, not the launch's."""
    sync = torch.cuda.synchronize if torch.cuda.is_available() else (lambda: None)
    fn(); sync(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); sync(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

---

# Part A · The Problem

## A1 · What is computed, and why an engineer wants it

Ex08's inverse problem found one number - the fins' effective coefficient - from one thermocouple, with everything else known. In a test laboratory two numbers are usually uncertain at once: the **heat capacity** of the base, because the alloy, the bonded layers and the thermal grease are not known exactly, and the **fan's coefficient**, because the fan's speed in the enclosure is not measured. Both shape the record of a thermocouple during a power pulse: the heat capacity sets how fast the temperature rises while the module is on, the coefficient how fast it falls once it is off.

An engineer who characterises sinks on a production line - every sink, every fan, every enclosure - wants the two numbers from each new record **at once**, not after a fit that runs the thermal model tens of times. This mini project builds **one network that reads a new noisy record and returns both numbers**, trained once over the whole range of the two, and compares it with the conventional answer: a least-squares fit with a finite-difference model.

## A2 · The heat sink, and what is new against Ex08

The heat sink and the pulse are Ex08 Part 2's: the 150 W module on for 20 s, a 150 s window, the thermocouple glued to the module face at $x$ = 80 mm, read every 2 s with 0.05 K of noise. The cell below draws it.

**What is new.** Two numbers are hidden instead of one: the base's volumetric heat capacity $\rho c_p$ and the fan's coefficient $h_\mathrm{fan}$. And the fins are cooled as in MP8.2A, by the fan **and by natural convection**:

$$h(\Delta T) = h_\mathrm{fan} + S \cdot C_\mathrm{nat}\left(\frac{\Delta T}{L_\mathrm{fin}}\right)^{1/4}$$

with $C_\mathrm{nat} = 1.42$ W/(m$^{7/4}$K$^{5/4}$) (laminar vertical plate in air, Incropera & DeWitt ch. 9, **assumed**) and $\Delta T$ the fin root's rise. The effective coefficient on the base, $h_\mathrm{eff}(\Delta T; h_\mathrm{fan})$, follows from Ex08's fin calculation. $S = 1$ is the strength of the change; at $S = 0$ the problem is Ex08's.

**The ranges** (assumed): $\rho c_p$ from 1.95 to 2.92 MJ/(m³K), ±20 % about Ex08's 2.43, and $h_\mathrm{fan}$ from 25 to 55 W/(m²K), about Ex08's 40.

In [ ]:
# The heat sink of Ex08, with the thermocouple, and the data of the problem
fig, ax = plt.subplots(figsize=(8, 3.6))
hs.draw_heatsink(ax, show_sensor=True)                    # Ex08's drawing with the thermocouple
plt.show()
rows = [("base plate", f"{hs.W_BASE*1e3:.0f} × {hs.T_BASE*1e3:.0f} mm aluminium, k = {hs.K_AL:.0f} W/(m·K)"),
        ("pulse", f"{hs.POWER:.0f} W for {hs.T_ON:.0f} s, window {hs.T_END:.0f} s"),
        ("thermocouple", f"x = {hs.SENSOR*1e3:.0f} mm on the module face, every {hs.DT_SENSOR:.0f} s, noise {mp.NOISE_K} K"),
        ("hidden ρc_p", f"{mp.RHO_CP_RANGE[0]/1e6:.2f} to {mp.RHO_CP_RANGE[1]/1e6:.2f} MJ/(m³K)"),
        ("hidden h_fan", f"{mp.H_FAN_RANGE[0]:.0f} to {mp.H_FAN_RANGE[1]:.0f} W/(m²K)"),
        ("natural convection", f"C_nat = {mp.C_NAT} W/(m^7/4 K^5/4), strength S = {mp.S_NAT}")]
for k, v in rows:
    print(f"{k:>18}  {v}")

**What you should see.** The sink with the module on top and the thermocouple beside it, 15 mm beyond the footprint's edge; and the data. Everything is Ex08's except the two ranges and the natural convection.

## A3 · The domain, the equation and its conditions

The domain is Ex08's base plate in cross-section, $0 < x < W$, $0 < z < t_b$, with the fins collapsed into $h_\mathrm{eff}$. The unknown is the rise above the air, $\theta(x, z, t)$.

$$\rho c_p\,\frac{\partial \theta}{\partial t} = k\left(\frac{\partial^2 \theta}{\partial x^2} + \frac{\partial^2 \theta}{\partial z^2}\right)$$

| where | condition | why it holds |
|---|---|---|
| module face, on the footprint ($z = t_b$) | $k\,\partial\theta/\partial z = q''\,s(t)$ | the module's 150 W while it is on, $s = 1$ for $t < 20$ s |
| module face, elsewhere | $\partial\theta/\partial z = 0$ | still air above the base |
| fin side ($z = 0$) | $k\,\partial\theta/\partial z = h_\mathrm{eff}(\theta; h_\mathrm{fan})\,\theta$ | the fins, fan and natural convection: **nonlinear** |
| ends ($x = 0$, $x = W$) | $\partial\theta/\partial x = 0$ | insulated |
| start | $\theta = 0$ | the sink starts at the air's temperature |

**What each hidden number does to the record.** While the module is on, the heat goes mostly into storage: the rise grows at a rate set by $\rho c_p$. Once it is off, the base gives its heat to the fins: the tail decays at a rate set by $h_\mathrm{eff}/(\rho c_p t_b)$. The record's rise therefore fixes $\rho c_p$, and its tail, together with $\rho c_p$, fixes $h_\mathrm{fan}$.

## A4 · Why there is no exact solution

Ex08's pulse has an exact series because its fin-side condition is linear in $\theta$. With natural convection the coefficient depends on the solution, superposition fails, and no series or closed form exists. The records are therefore made by the **converged finite-difference march** (part C), checked at $S = 0$ against Ex08's exact pulse.

In [ ]:
# The five test records, observed (with noise) and the noise-free curves (hidden)
G = np.load("MP8.2B_truth.npz")
fig, ax = plt.subplots(figsize=(10, 3.6))
for i in range(5):
    l, = ax.plot(G["t"], G[f"readings_{i}"], ".", ms=3)
    ax.plot(G["t"], G[f"clean_{i}"], "-", lw=0.6, color=l.get_color(),
            label=f"record {i}: ρc_p {G['hidden_rho_cp'][i]/1e6:.2f} MJ/(m³K), h_fan {G['hidden_h_fan'][i]:.0f} W/(m²K)")
ax.set_xlabel("t [s]"); ax.set_ylabel("thermocouple rise [K]"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

**What you should see.** Five records: each rises while the module is on, peaks a few seconds after the switch-off (the heat needs time to spread the 15 mm to the thermocouple), and decays. A lower $\rho c_p$ (records 1 and 3) gives a steeper rise and a higher peak; a higher $h_\mathrm{fan}$ (records 2 and 3) a faster decay. The hidden values are printed here because this notebook is the solved story; the network never sees them for these five records.

## A5 · What is asked

| quantity | for | required |
|---|---|---|
| $\rho c_p$ and $h_\mathrm{fan}$ | the five test records, which you do not train on | both within **5 %** of the hidden values |
| the time to answer a new record online | one Colab T4 runtime | at least **10 times shorter** than a new least-squares fit with the reference model (part C measures it) |

**One network, trained once**, over the whole range of the two hidden numbers: it answers any new record without training again. This is an **amortised** inverse problem: the cost of fitting is paid once, offline, and every new record costs one evaluation. The training time is reported apart, as the offline cost.

---

# Part B · The Data and the Physics

## B1 · What the network takes and returns

Two designs answer the question, and both are physics-based:

* **a network from the record to the numbers**, $(\theta_1, \dots, \theta_{75}) \mapsto (\rho c_p, h_\mathrm{fan})$, trained on many records that the physics generates for random hidden numbers, with the thermocouple's noise added afresh during training. The physics enters through the records: each is a solution of the equation above. A new record costs one evaluation.
* **a physics-informed network conditioned on the numbers**, $(x, z, t, \rho c_p, h_\mathrm{fan}) \mapsto \theta$, trained on the residuals of the equation over the whole range of the two numbers; a new record is then answered by a fast fit through the network (Gauss-Newton on two numbers, the network's derivatives with respect to them by automatic differentiation). A new record costs a few evaluations.

The first is the more direct for the question asked, and is the solved design. The second is closer to Ex08 and is worth trying if you want the field as well as the numbers. Either way the outputs are best returned as **logarithms scaled to $-1$ to $1$** over the ranges: both numbers are positive, and their ranges differ by five orders of magnitude.

In [ ]:
# What a hidden number does to the record: the change for +5 % in each, from Ex08's values
r0 = mp.record(hs.RHO_CP, 40.0, mp.TRUTH_GRIDS[1])
r_rho = mp.record(hs.RHO_CP * 1.05, 40.0, mp.TRUTH_GRIDS[1])
r_h = mp.record(hs.RHO_CP, 42.0, mp.TRUTH_GRIDS[1])
r_s0 = mp.record(hs.RHO_CP, 40.0, mp.TRUTH_GRIDS[1], s_nat=0.0)
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(mp.T_READ, r_rho - r0, label="ρc_p + 5 %"); ax.plot(mp.T_READ, r_h - r0, label="h_fan + 5 %")
ax.plot(mp.T_READ, r_s0 - r0, label="no natural convection (S = 0)")
ax.axhspan(-mp.NOISE_K, mp.NOISE_K, color="grey", alpha=0.2, label="± noise")
ax.set_xlabel("t [s]"); ax.set_ylabel("change of the record [K]"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f"+5 % ρc_p: largest change {np.abs(r_rho - r0).max():.2f} K at {mp.T_READ[np.argmax(np.abs(r_rho - r0))]:.0f} s")
print(f"+5 % h_fan: largest change {np.abs(r_h - r0).max():.2f} K at {mp.T_READ[np.argmax(np.abs(r_h - r0))]:.0f} s")
print(f"natural convection lowers the record by up to {np.abs(r_s0 - r0).max():.2f} K")

**What you should see.** A 5 % change of $\rho c_p$ moves the record by up to 0.44 K, at the switch-off (20 s); a 5 % change of $h_\mathrm{fan}$ by up to 0.17 K, in the tail near 78 s. Both are several times the 0.05 K noise, summed over 75 readings: 5 % is resolvable. Natural convection lowers the record by up to 0.49 K: a model without it would mistake it for a higher fan coefficient.

## B2 · The physics the network is trained with

For the first design, the physics is in the **training records**: `mp.training_records(n)` draws $n$ pairs uniformly over the ranges and marches each on 41 × 5 nodes with a 0.5 s step - within 0.01 K of the ground truth (part C) and a few tens of milliseconds each. A training set of 2 000 records takes one to three minutes of CPU (96 s in the solved run).

| term | what it enforces |
|---|---|
| $\mathcal{L}_\mathrm{fit}$ | the network's numbers, as scaled logarithms, against the true numbers of each training record |
| the noise, drawn afresh at every step | the network learns to read through 0.05 K of noise, not to memorise records |

For the second design the loss terms are those of Ex08 Part 2 with the nonlinear fin side, as in MP8.2A, with $\rho c_p$ and $h_\mathrm{fan}$ as inputs.

**What can be built in.** The scaled logarithm for the outputs; the record's own normalisation (subtract the mean training record and divide by the spread at each instant, so every input is of order one). The noise level is known (0.05 K) and is part of the physics of the measurement.

## B3 · The training records

The cell below builds 2 000 training records (seed 82) and shows the spread of the pairs and of the records. The five test pairs of part A are not among them on purpose.

In [ ]:
# 2 000 training records from the physics (one to three minutes on one CPU core)
t0 = time.perf_counter()
TRAIN = dict(zip(("rho_cp", "h_fan", "records"), mp.training_records(2000, seed=82)))
print(f"built in {time.perf_counter() - t0:.0f} s")
fig, ax = plt.subplots(1, 2, figsize=(13, 3.4))
ax[0].scatter(TRAIN["rho_cp"] / 1e6, TRAIN["h_fan"], s=2, alpha=0.4, label="training pairs")
ax[0].scatter(G["hidden_rho_cp"] / 1e6, G["hidden_h_fan"], marker="*", s=150, color="r", label="test pairs")
ax[0].set_xlabel("ρc_p [MJ/(m³K)]"); ax[0].set_ylabel("h_fan [W/(m²K)]"); ax[0].legend(fontsize=8)
ax[1].plot(mp.T_READ, TRAIN["records"][:60].T, lw=0.4, color="tab:blue", alpha=0.5)
ax[1].set_xlabel("t [s]"); ax[1].set_ylabel("rise [K]"); ax[1].set_title("sixty training records", fontsize=9)
plt.tight_layout(); plt.show()

**What you should see.** One to three minutes of CPU (96 s in the solved run, 170 s on a slower laptop). Left, the pairs fill the rectangle of the ranges; the five test pairs are red stars inside it. Right, sixty records: peaks from about 8 to 12 K and tails from about 1 to 3 K at 150 s.

---

# Part C · The Reference Model and the Ground Truth

## C1 · Three solvers, three jobs

| | what it is | grid | its job |
|---|---|---|---|
| the **reference model** | Crank-Nicolson finite differences, the nonlinear fin side by chord iterations | 21 × 3 nodes, a step of 1 s | the forward model of the least-squares reference |
| the **least-squares reference** | `scipy.optimize.least_squares` on the logarithms of the two numbers, the reference model as the forward model | - | the conventional inverse answer, **timed** per new record |
| the **ground truth** | the same march | 161 × 17 nodes, a step of 0.125 s | makes the test records and **scores** the answers |

The march is `mp82b.March`, MP8.2A's scheme for one pulse: central differences with ghost nodes, Crank-Nicolson in time with two backward-Euler half steps after each switch (Rannacher's start), and three chord iterations per step on the nonlinear fin side.

In [ ]:
# The ground truth: convergence at Ex08's values, and Ex08's limit, from the truth file
print("grid (nx × nz, dt)      peak of the record   at 150 s    largest change from the grid before")
prev = None
for g, r in zip(G["conv_grids"], G["conv_records"]):
    ch = "" if prev is None else f"{np.abs(r - prev).max():.5f} K"
    print(f"{int(g[0]):4d} × {int(g[1]):3d}, {g[2]:.3f} s     {r.max():10.4f} K     {r[-1]:.4f} K    {ch}")
    prev = r
d = [np.abs(G["conv_records"][i + 1] - G["conv_records"][i]).max() for i in range(3)]
print("ratios of the changes:", np.round(np.array(d[:-1]) / np.array(d[1:]), 2))
print(f"finest grid against the Richardson extrapolation: {np.abs(G['conv_records'][-1] - G['conv_extrapolated']).max():.5f} K")
print(f"at S = 0, against Ex08's exact pulse at the thermocouple: {float(G['zero_strength_error_K']):.5f} K at most")

**What you should see.** The record changes by 0.022, 0.0045 and 0.0011 K from grid to grid: falling by about four, second order. The finest grid is within 0.0004 K of the extrapolation - a hundred times smaller than the 0.05 K noise, so the test records' error is the noise, not the solver. At $S = 0$ the march reproduces Ex08's exact pulse to 4e-04 K.

## C2 · The least-squares reference: accuracy and time

For each test record, the least-squares reference starts from Ex08's values, runs the reference model for each trial pair and for each finite-difference derivative, and stops when the pair changes by less than 1e-06 in its logarithm. Each run of the reference model builds and factorises its matrices for the trial pair: both numbers change them.

In [ ]:
# The least-squares reference on the five test records
worst = np.zeros(2); runs, secs = [], []
for i in range(5):
    f = mp.lsq_fit(G[f"readings_{i}"])
    e = np.array([f["rho_cp"] / G["hidden_rho_cp"][i] - 1, f["h_fan"] / G["hidden_h_fan"][i] - 1]) * 100
    worst = np.maximum(worst, np.abs(e)); runs.append(f["runs"]); secs.append(f["seconds"])
    print(f"record {i}: ρc_p {f['rho_cp']/1e6:.3f} MJ/(m³K) ({e[0]:+.2f} %), h_fan {f['h_fan']:.2f} W/(m²K) ({e[1]:+.2f} %), "
          f"{f['runs']} runs, {f['seconds']:.2f} s, misfit {f['rms']:.3f} K")
t_lsq = float(np.median(secs))
print(f"\nleast squares: worst {worst[0]:.2f} % on ρc_p, {worst[1]:.2f} % on h_fan; {t_lsq:.2f} s per new record (median)")
print(f"the network must answer a new record in under {t_lsq*1e3/10:.0f} ms (ten times shorter)")
REF = {"t_lsq_s": t_lsq, "worst_lsq_pct": worst, "runs": runs}   # kept for part E

**What you should see.** The least-squares reference finds both numbers within about 0.4 % on every record, with a misfit of 0.05 K - the noise: the model and the data agree. It needs 9 to 21 runs of the reference model and 0.3 to 1.2 s per record depending on the CPU (a median of 0.35 s in the solved run). That time is the bar: the network must answer in under a tenth of it, about 35 ms here.

---

# Part D · The Physics-Informed Network

## D1 · What is given, and what is yours

**Yours** are three things, each a TODO cell: the **network** (TODO 1), its **training** (TODO 2), and the **figures** (TODO 3).

**The contract.** Whatever your design, provide a function `estimate(readings)` that takes one record - 75 readings in kelvin, a NumPy array - and returns `(rho_cp, h_fan)` in J/(m³K) and W/(m²K). It is what part E scores and times.

## D2 · What is given

`TRAIN` (part B): 2 000 noise-free training records and their pairs. The helpers below scale the pairs to and from $-1..1$ in their logarithms, and `quick_score()` gives the worst error in per cent on 200 validation records with fresh noise - records the network has not trained on, from the same physics.

In [ ]:
# Given: scaling of the pairs, a validation set, and a quick score ----------------
LO = np.log([mp.RHO_CP_RANGE[0], mp.H_FAN_RANGE[0]])     # the logarithms of the ranges' ends
HI = np.log([mp.RHO_CP_RANGE[1], mp.H_FAN_RANGE[1]])
def to_unit(rho_cp, h_fan):
    """the pairs as their logarithms scaled to -1..1: the network's natural outputs, shape (n, 2)"""
    return (np.log(np.c_[rho_cp, h_fan]) - LO) / (HI - LO) * 2 - 1
def from_unit(y):
    """back to (rho_cp, h_fan) in SI units"""
    v = np.exp((np.asarray(y) + 1) / 2 * (HI - LO) + LO)
    return v[..., 0], v[..., 1]

VAL = dict(zip(("rho_cp", "h_fan", "records"), mp.training_records(200, seed=7, verbose=False)))   # a validation set
VAL["noisy"] = VAL["records"] + mp.NOISE_K * np.random.default_rng(8).standard_normal(VAL["records"].shape)
def quick_score():
    """the worst error (%) of ρc_p and of h_fan over the 200 validation records with noise"""
    est = np.array([estimate(r) for r in VAL["noisy"]])
    return tuple(np.abs(est / np.c_[VAL["rho_cp"], VAL["h_fan"]] - 1).max(0) * 100)

## D3 · TODO 1 · Your network, and `estimate`

Write the network and `estimate(readings)`. Decide the design (part B, section 1), the inputs' normalisation, the outputs, the size. Remember that the inputs are 75 numbers of very different sizes - the peak is ten times the tail - and the outputs two numbers whose ranges differ by five orders of magnitude.

In [ ]:
# TODO 1 - your network, and estimate(readings) -> (rho_cp, h_fan)
import torch.nn as nn
...

def estimate(readings):
    ...                                                  # return (rho_cp in J/(m³K), h_fan in W/(m²K))

## D4 · TODO 2 · Your training

Train on `TRAIN`. Decide the loss, the optimiser, the steps, the batch, and how the noise enters. Keep the loss history for TODO 3 and **measure the training time** in `TRAIN_SECONDS` - and say in the report whether you count the three minutes of building `TRAIN` in it (the solved form reports both).

In [ ]:
# TODO 2 - your training
history = []
t0 = time.perf_counter()
...
TRAIN_SECONDS = time.perf_counter() - t0
print(f"trained in {TRAIN_SECONDS:.0f} s; worst validation error: ρc_p %.2f %%, h_fan %.2f %%" % quick_score())

## D5 · TODO 3 · Your figures

At least: the loss during training, and the estimate against the truth for the validation records, for both numbers. Save each figure you want in the report as `MP8.2B_report_<name>.png`.

In [ ]:
# TODO 3 - your figures
...

## D6 · The evaluation, given

This cell runs `estimate` on the five test records and keeps the answers in `R`; it also times `estimate` on one new record against the least-squares reference's median time from part C, both on this runtime, and asks the network for the same five records with ten fresh noise seeds each, to see how the noise moves its answer.

In [ ]:
# The evaluation, given ------------------------------------------------------------
out = {}
est = np.array([estimate(G[f"readings_{i}"]) for i in range(5)])
out["rho_cp"], out["h_fan"] = est[:, 0], est[:, 1]
rng = np.random.default_rng(2026)                         # ten fresh noise seeds per test record
spread = np.array([[estimate(G[f"clean_{i}"] + mp.NOISE_K * rng.standard_normal(75)) for _ in range(10)] for i in range(5)])
out["spread_rho_cp"], out["spread_h_fan"] = spread[..., 0], spread[..., 1]
out["t_net"] = np.array(middle(lambda: estimate(G["readings_0"])))
out["t_lsq"] = np.array(REF["t_lsq_s"])
out["train_seconds"] = np.array(TRAIN_SECONDS)
R = out                                                  # part E reads the answers from here
for i in range(5):
    print(f"record {i}: ρc_p {est[i,0]/1e6:.3f} MJ/(m³K) ({100*(est[i,0]/G['hidden_rho_cp'][i]-1):+.2f} %), "
          f"h_fan {est[i,1]:.2f} W/(m²K) ({100*(est[i,1]/G['hidden_h_fan'][i]-1):+.2f} %)")
print(f"online, one new record: network {out['t_net']*1e3:.3f} ms, least squares {out['t_lsq']*1e3:.0f} ms "
      f"({out['t_lsq']/out['t_net']:.0f} times)")

---

# Part E · The Report

## E1 · What this part does

It scores your network's answers, kept by section D6 in `R`, against the hidden values in `G` and the least-squares reference's time `REF` from part C: the goal metrics (TODO 4) and the report's figures (TODO 5). Then come the eighteen questions.

In [ ]:
# What part E uses, all from the parts above
G = np.load("MP8.2B_truth.npz")
print("in R:", ", ".join(R))
print(f"least squares (part C): worst {REF['worst_lsq_pct'][0]:.2f} % and {REF['worst_lsq_pct'][1]:.2f} %, {REF['t_lsq_s']*1e3:.0f} ms per record")

## E2 · TODO 4 · The goal metrics

Compute and put in `METRICS`: the error (%) of $\rho c_p$ and of $h_\mathrm{fan}$ on each test record, the worst of each, the spread (standard deviation, %) of each over the ten noise seeds, the online time of the network and of the least squares, their ratio, and the training time. Say in one line each whether the requirements of part A are met.

In [ ]:
# TODO 4 - the goal metrics
METRICS = {}
...
for k, v in METRICS.items():
    print(f"{k:>45}: {v:.2f}")

## E3 · TODO 5 · The report's figures

Draw and save as `MP8.2B_report_<name>.png`: the five test records with the record your estimate predicts (`mp.record(rho_cp, h_fan, mp.TRUTH_GRIDS[1])`) and the misfit; the estimates with their spread over the noise seeds against the hidden values; the online times.

In [ ]:
# TODO 5 - the report's figures
...

## E4 · The report

Answer the eighteen questions below, each in a few sentences **with your numbers**; an answer needs at least fifteen words. Questions 1 to 10 are about your network and its result; 11 to 17 ask you to use it; 18 concludes. The next cell writes `MP8.2B_report.md` with the results table above and your answers, and the cell after it turns the report into a PDF. **Upload the PDF.**

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "1. Your design: a network from the record to the numbers, a network conditioned on the numbers with a fit through it, or another? Why?": """
""",
    "2. Your network: its inputs and their normalisation, layers, neurons, outputs and number of weights. Why this size?": """
""",
    "3. Where does the physics enter your design? Name every place: the equation, the conditions, the noise, the ranges.": """
""",
    "4. Your training: the number of training records and how they were made, the optimiser, steps, batch and learning rate, and the offline time (building the records and training) in seconds.": """
""",
    "5. How does the noise enter your training, and what happens to the test errors if the network is trained on noise-free records only?": """
""",
    "6. The two numbers on the five test records: your error on each in per cent. Are all within 5 %? Compare with the least-squares reference.": """
""",
    "7. On which record is your error largest, and why there? Use the record's hidden values and the sensitivities of part B.": """
""",
    "8. The spread of your estimates over ten noise seeds per record: how large for each number, and how does it compare with your error?": """
""",
    "9. The online time for one new record: your network against the least-squares reference on the same runtime, and the ratio. What does each spend its time on?": """
""",
    "10. How many records must be answered before the offline cost is repaid, counting the building of the training records?": """
""",
    "11. Which of the two numbers is the better determined by the record, and why? Use the change of the record for +5 % in each (part B).": """
""",
    "12. The records your estimates predict against the readings: how large is the misfit, and is it the noise alone?": """
""",
    "13. Outside the range: make a record with ρc_p = 3.2 MJ/(m³K) or h_fan = 65 W/(m²K) and ask your network. How large is the error, and why?": """
""",
    "14. If the model left out natural convection but the sink had it, how far off would h_fan be? Fit record 0 with a forward model at S = 0 and give both numbers.": """
""",
    "15. Train again with another seed. How much do the worst errors of the two runs differ? Is your result reproducible within 5 %?": """
""",
    "16. With half the readings (every 4 s) or twice the noise (0.1 K), what happens to your errors? Retrain if your design needs it, and say what it cost.": """
""",
    "17. If the thermocouple were moved to x = 50 mm, under the module, could your network answer? What would you change?": """
""",
    "18. To conclude: for which engineering task would you use your network rather than the least-squares reference, given your measured accuracy, online time and offline time? For which would you not?": """
""",
}

In [ ]:
MIN_WORDS = 15                                              # an answer is a few sentences with numbers, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:

    rows = ["| metric | value |", "|---|---|"] + [f"| {k} | {v:.2f} |" for k, v in METRICS.items()]
    out = ["# MP8.2B — Heat Capacity and Coefficient Together", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("MP8.2B_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figures saved as `MP8.2B_report*.png`, and downloads it.

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns MP8.2B_report.md into MP8.2B_report.pdf, with any figure
# saved as MP8.2B_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "MP8.2B_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "MP8.2B_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("MP8.2B_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "MP8.2B_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass